# LLM benchmark

Tables 4 and 5. Scores the runs written by `scripts/run_llm_benchmark.py`; makes no API calls.

In [ ]:
import sys
sys.path.insert(0, "..")

from embodiment import config as cfg

import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score

RESULTS = cfg.BENCHMARK_DIR


def load(path):
    df = pd.read_csv(path)
    df["pred"] = df["pred"].fillna("__invalid__")   # unparseable counts as wrong
    return df


runs = {p.stem: load(p) for p in sorted(RESULTS.glob("*__*.csv"))
        if not p.name.startswith("_")}

# The no-context runs cover more sentences than the context runs, which need a
# recoverable surrounding passage. Everything is scored on the intersection.
common = set.intersection(*(set(zip(d["filename"], d["sentence_id"]))
                            for d in runs.values()))
print(f"{len(runs)} runs, {len(common)} sentences in common")


def restrict(df):
    return df[[(f, s) in common
               for f, s in zip(df["filename"], df["sentence_id"])]]

## Baselines

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import FeatureUnion, Pipeline

gold = pd.read_csv(cfg.GOLD)


def tfidf():
    return FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True,
                                 min_df=2)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5),
                                 sublinear_tf=True, min_df=2)),
    ])


BASELINES = {
    "majority": DummyClassifier(strategy="most_frequent"),
    "naive-bayes": Pipeline([("tfidf", tfidf()), ("clf", MultinomialNB())]),
    "logreg-tfidf": Pipeline([("tfidf", tfidf()),
                              ("clf", LogisticRegression(
                                  max_iter=2000, class_weight="balanced"))]),
}

# The fine-tuned model and the classical baselines both trained on most of these
# sentences, so both are scored out-of-fold rather than run directly.
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=cfg.SEED)
for name, model in BASELINES.items():
    path = RESULTS / f"{name}__baseline.csv"
    if path.exists() and len(pd.read_csv(path)) == len(common):
        continue
    oof = cross_val_predict(model, gold["text"], gold["label"], cv=skf)
    out = gold[["filename", "sentence_id", "text"]].copy()
    out["gold"], out["pred"] = gold["label"], oof
    restrict(out).to_csv(path, index=False)
    print(f"wrote {path.name}")

rb = RESULTS / "roberta__cv.csv"
if not (rb.exists() and len(pd.read_csv(rb)) == len(common)):
    cv = restrict(pd.read_csv(cfg.CV_PREDICTIONS))
    cv[["filename", "sentence_id", "text", "gold", "pred"]].to_csv(rb,
                                                                   index=False)
    print(f"wrote {rb.name}")

## Table 4

In [ ]:
rows = []
for path in sorted(RESULTS.glob("*__*.csv")):
    df = restrict(load(path))
    model, condition = path.stem.split("__")
    rows.append({
        "model": model, "condition": condition, "n": len(df),
        "accuracy": accuracy_score(df["gold"], df["pred"]),
        "macro_f1": f1_score(df["gold"], df["pred"], labels=cfg.LABELS,
                             average="macro", zero_division=0),
        "kappa": cohen_kappa_score(df["gold"], df["pred"]),
        "invalid": (df["pred"] == "__invalid__").mean(),
    })

table4 = pd.DataFrame(rows).sort_values("accuracy").reset_index(drop=True)
cfg.TABLES.mkdir(exist_ok=True)
table4.to_csv(cfg.TABLES / "table4_accuracy.csv", index=False)
table4.round(3)

## Table 5

In [ ]:
# One column per system, on the few-shot with-context condition.
COLUMNS = [("gpt-5.1", "fewshot_context"), ("gpt-5-mini", "fewshot_context"),
           ("llama3.3-70b-local", "fewshot_context"),
           ("qwen3-32b-local", "fewshot_context"), ("roberta", "cv")]

table5 = pd.DataFrame({"label": cfg.LABELS})
table5["n"] = [int((restrict(load(RESULTS / "roberta__cv.csv"))["gold"] == l).sum())
               for l in cfg.LABELS]
for model, condition in COLUMNS:
    df = restrict(load(RESULTS / f"{model}__{condition}.csv"))
    table5[model] = [f1_score(df["gold"], df["pred"], labels=[l],
                              average="macro", zero_division=0)
                     for l in cfg.LABELS]

macro = {m: f1_score(*[restrict(load(RESULTS / f"{m}__{c}.csv"))[k]
                       for k in ("gold", "pred")],
                     labels=cfg.LABELS, average="macro", zero_division=0)
         for m, c in COLUMNS}
micro = {m: accuracy_score(*[restrict(load(RESULTS / f"{m}__{c}.csv"))[k]
                             for k in ("gold", "pred")])
         for m, c in COLUMNS}
table5 = table5.sort_values("label").reset_index(drop=True)
table5.loc[len(table5)] = {"label": "Macro-F1", "n": None, **macro}
table5.loc[len(table5)] = {"label": "Micro-F1", "n": None, **micro}
table5.to_csv(cfg.TABLES / "table5_per_label.csv", index=False)
table5.round(2)

## Paired comparisons

In [ ]:
from scipy.stats import binomtest


def mcnemar(stem_a, stem_b):
    """Sentences one system gets right and the other wrong, both directions."""
    a, b = load(RESULTS / f"{stem_a}.csv"), load(RESULTS / f"{stem_b}.csv")
    m = a.merge(b, on=["filename", "sentence_id"], suffixes=("_a", "_b"))
    ok_a, ok_b = m["pred_a"] == m["gold_a"], m["pred_b"] == m["gold_a"]
    only_a, only_b = int((ok_a & ~ok_b).sum()), int((~ok_a & ok_b).sum())
    p = binomtest(only_a, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    print(f"{stem_a} ({ok_a.mean():.3f}) vs {stem_b} ({ok_b.mean():.3f}): "
          f"{only_a} vs {only_b}, p = {p:.2g}")


mcnemar("qwen3-32b-local__fewshot_context", "gpt-5-mini__fewshot_context")
mcnemar("roberta__cv", "qwen3-32b-local__fewshot_context")

# few-shot against zero-shot, per model
for model in ["gpt-5-mini", "gpt-5.1", "qwen3-32b-local", "llama3.3-70b-local"]:
    mcnemar(f"{model}__fewshot_context", f"{model}__zeroshot_context")